In [1]:
AminoAcidMass = {'G': 57, 'A': 71, 'S': 87, 'P': 97, 'V': 99, 'T': 101, 'C': 103, 'I': 113,
                 'L': 113, 'N': 114, 'D': 115, 'K': 128, 'Q': 128, 'E': 129, 'M': 131,
                 'H': 137, 'F': 147, 'R': 156, 'Y': 163, 'W': 186}

def LinearSpectrum(Peptide, AminoAcidMass):
    """
    计算线性肽的质谱
    
    参数:
    Peptide (str): 氨基酸序列字符串
    AminoAcidMass (dict): 氨基酸到质量的映射字典
    
    返回:
    list: 排序后的质谱列表
    """
    n = len(Peptide)
    PrefixMass = [0] * (n + 1)
    
    # 计算前缀质量（使用字典直接验证）
    for i in range(1, n + 1):
        aa = Peptide[i - 1]
        if aa not in AminoAcidMass:
            raise ValueError(f"氨基酸 '{aa}' 的质量未定义")
        PrefixMass[i] = PrefixMass[i - 1] + AminoAcidMass[aa]
    
    # 生成质谱
    LinearSpectrum = [0]
    for i in range(n):
        for j in range(i + 1, n + 1):
            mass = PrefixMass[j] - PrefixMass[i]
            LinearSpectrum.append(mass)
    
    return sorted(LinearSpectrum)

def CyclicSpectrum(Peptide, AminoAcidMass):
    """
    计算环肽的质谱
    
    参数:
    Peptide (str): 氨基酸序列字符串
    AminoAcidMass (dict): 氨基酸到质量的映射字典
    
    返回:
    list: 排序后的唯一质谱值列表
    """
    n = len(Peptide)
    total_mass = sum(AminoAcidMass[aa] for aa in Peptide)
    
    # 初始化前缀质量数组
    PrefixMass = [0] * (n + 1)
    for i in range(1, n + 1):
        aa = Peptide[i - 1]
        if aa not in AminoAcidMass:
            raise ValueError(f"氨基酸 '{aa}' 的质量未定义")
        PrefixMass[i] = PrefixMass[i - 1] + AminoAcidMass[aa]
    
    # 初始化质谱列表（只包含0）
    spectrum_list = [0]
    
    # 1. 添加所有线性片段（包括整个肽）
    for i in range(n):
        for j in range(i + 1, n + 1):
            mass = PrefixMass[j] - PrefixMass[i]
            spectrum_list.append(mass)
    
    # 2. 正确添加跨越末端的片段（仅非连续部分）
    for i in range(1, n):  # 从1开始，避免重复0
        for j in range(i + 1, n):
            # 计算跨越末端的片段：从j到末尾 + 从开头到i-1
            mass = (PrefixMass[n] - PrefixMass[j]) + PrefixMass[i]
            spectrum_list.append(mass)
    
    # 去重并排序
    return sorted(spectrum_list)

In [4]:
Peptide = "PEEP"
theoretical_result = LinearSpectrum(Peptide, AminoAcidMass)
theoretical_str = " ".join(map(str, theoretical_result))
print(theoretical_str)

0 97 97 129 129 226 226 258 355 355 452


In [6]:
def Score(Peptide, Spectrum):
    # 1. 将字符串分割为整数列表
    peptide_values = list(map(int, Peptide.split()))
    spectrum_values = list(map(int, Spectrum.split()))
    
    # 2. 统计理论值中每个质量的出现次数
    from collections import defaultdict
    peptide_counts = defaultdict(int)
    for value in peptide_values:
        peptide_counts[value] += 1
    
    score = 0
    # 3. 遍历测量值，匹配理论值中未耗尽的质量
    for value in spectrum_values:
        if peptide_counts[value] > 0:
            score += 1
            peptide_counts[value] -= 1
    
    return score

# 测试示例
peptide = "0 97 97 129 129 226 226 258 355 355 452"
spectrum = "0 97 97 129 129 194 203 226 226 258 323 323 323 355 403 452"
print(Score(peptide, spectrum))  # 输出：11，与示例结果一致

10


In [1]:
from collections import Counter

# 20种氨基酸的质量列表
AMINO_ACID_MASSES = [57, 71, 87, 97, 99, 101, 103, 113, 114, 115, 128, 129, 131, 137, 147, 156, 163, 186]

def mass(peptide):
    """计算肽段的总质量（氨基酸质量之和）"""
    return sum(peptide)

def parent_mass(spectrum):
    """谱图中最大质量值即为父质量"""
    return max(spectrum)

def get_linear_spectrum(peptide):
    """生成肽段的线性谱图"""
    n = len(peptide)
    prefix_sum = [0] * (n + 1)
    # 计算前缀和
    for i in range(1, n + 1):
        prefix_sum[i] = prefix_sum[i - 1] + peptide[i - 1]
    
    spectrum = [0]  # 总是包含0
    # 计算所有可能的子串质量
    for i in range(n):
        for j in range(i + 1, n + 1):
            spectrum.append(prefix_sum[j] - prefix_sum[i])
    return sorted(spectrum)

def get_cyclic_spectrum(peptide):
    """生成肽段的循环谱图"""
    n = len(peptide)
    if n == 0:
        return [0]
    
    prefix_sum = [0] * (n + 1)
    for i in range(1, n + 1):
        prefix_sum[i] = prefix_sum[i - 1] + peptide[i - 1]
    
    total_mass = prefix_sum[n]
    spectrum = [0, total_mass]  # 包含0和完整肽段质量
    
    # 考虑循环片段：长度从1到n-1
    extended_peptide = peptide + peptide[:-1]  # 复制肽段（去掉最后一个氨基酸）
    for start in range(n):
        for length in range(1, n):
            end = start + length
            if end <= len(extended_peptide):
                # 使用前缀和计算片段质量
                subpep_sum = prefix_sum[start + length] - prefix_sum[start] if start + length <= n else None
                if subpep_sum is None:
                    # 处理循环部分
                    fragment = extended_peptide[start:end]
                    subpep_sum = sum(fragment)
                spectrum.append(subpep_sum)
    return sorted(spectrum)

def score_linear(peptide, exp_spectrum):
    """计算肽段线性谱图与实验谱图的匹配得分"""
    theo_spectrum = get_linear_spectrum(peptide)
    theo_counter = Counter(theo_spectrum)
    exp_counter = Counter(exp_spectrum)
    score = 0
    for mass_val in theo_counter:
        if mass_val in exp_counter:
            score += min(theo_counter[mass_val], exp_counter[mass_val])
    return score

def score_cyclic(peptide, exp_spectrum):
    """计算肽段循环谱图与实验谱图的匹配得分"""
    theo_spectrum = get_cyclic_spectrum(peptide)
    theo_counter = Counter(theo_spectrum)
    exp_counter = Counter(exp_spectrum)
    score = 0
    for mass_val in theo_counter:
        if mass_val in exp_counter:
            score += min(theo_counter[mass_val], exp_counter[mass_val])
    return score

def expand(leaderboard):
    """扩展候选肽段列表：每个肽段添加20种氨基酸"""
    new_leaderboard = []
    for pep in leaderboard:
        for aa_mass in AMINO_ACID_MASSES:
            new_pep = pep + (aa_mass,)  # 元组不可变，使用元组表示肽段
            new_leaderboard.append(new_pep)
    return new_leaderboard

def trim(leaderboard, spectrum, n):
    """修剪候选肽段列表：保留线性谱图得分最高的前N个（含并列）"""
    if len(leaderboard) <= n:
        return leaderboard
    
    # 计算每个肽段的线性得分并排序
    scores = [score_linear(pep, spectrum) for pep in leaderboard]
    sorted_peps = sorted(zip(scores, leaderboard), reverse=True)
    
    # 确定前N名的得分阈值
    threshold_score = sorted_peps[n][0]
    trimmed = []
    for score, pep in sorted_peps:
        if score >= threshold_score:
            trimmed.append(pep)
        else:
            break  # 已低于阈值，提前结束
    return trimmed

def leaderboard_cyclopeptide_sequencing(spectrum, n):
    """主函数：执行Leaderboard Cyclopeptide Sequencing算法"""
    leaderboard = [()]  # 初始为空肽段（元组表示）
    leader_peptide = ()  # 当前最佳肽段
    pm = parent_mass(spectrum)  # 目标父质量
    
    while leaderboard:
        # 1. 扩展所有候选肽段
        leaderboard = expand(leaderboard)
        
        # 2. 筛选肽段并更新最佳肽段
        new_leaderboard = []
        for pep in leaderboard:
            pep_mass = mass(pep)
            if pep_mass == pm:
                # 计算循环得分并更新最佳肽段
                current_score = score_cyclic(pep, spectrum)
                leader_score = score_cyclic(leader_peptide, spectrum) if leader_peptide else -1
                if current_score > leader_score:
                    leader_peptide = pep
                new_leaderboard.append(pep)  # 保留在列表中
            elif pep_mass < pm:
                new_leaderboard.append(pep)  # 质量不足，继续保留
            # 质量超过pm的肽段被丢弃
        leaderboard = new_leaderboard
        
        # 3. 修剪候选列表
        if leaderboard:
            leaderboard = trim(leaderboard, spectrum, n)
    
    return leader_peptide

# 示例用法
if __name__ == "__main__":
    # 示例谱图（需替换为实际数据）
    spectrum_str = "0 71 113 129 147 200 218 260 313 331 347 389 460"
    spectrum_example = list(map(int, spectrum_str.split()))
    n_example = 10
    
    best_peptide = leaderboard_cyclopeptide_sequencing(spectrum_example, n_example)
    # 输出格式：质量序列用"-"连接
    print("-".join(str(mass) for mass in best_peptide))

147-71-129-113


In [2]:
# 示例用法
if __name__ == "__main__":
    # 示例谱图（需替换为实际数据）
    spectrum_str = "0 71 71 71 87 97 97 99 103 113 113 113 113 114 115 128 129 129 147 147 156 174 186 186 186 200 210 210 211 212 212 218 232 241 241 242 257 257 257 260 262 269 270 273 276 299 303 321 324 325 328 338 344 345 347 347 354 359 359 366 367 370 372 375 383 386 416 418 443 443 446 450 450 451 452 457 472 472 473 474 480 480 480 481 483 485 514 521 530 533 544 556 561 563 565 571 577 579 580 585 586 593 595 601 604 604 608 628 629 632 643 650 657 658 675 683 684 690 691 692 692 694 700 703 707 721 721 730 741 742 742 747 754 763 771 778 790 797 804 805 813 818 818 820 829 834 836 839 840 844 850 855 861 868 877 889 891 901 907 915 927 931 933 933 942 948 949 952 952 953 954 957 960 963 964 976 1004 1004 1015 1020 1024 1028 1030 1041 1046 1047 1051 1051 1060 1065 1066 1075 1077 1080 1089 1089 1093 1101 1101 1113 1137 1138 1144 1148 1150 1160 1163 1164 1171 1172 1175 1180 1184 1190 1193 1193 1204 1206 1214 1215 1227 1234 1236 1251 1261 1263 1278 1279 1284 1285 1287 1292 1293 1298 1300 1301 1306 1307 1322 1324 1334 1349 1351 1358 1370 1371 1379 1381 1392 1392 1395 1401 1405 1410 1413 1414 1421 1422 1425 1435 1437 1441 1447 1448 1472 1484 1484 1492 1496 1496 1505 1508 1510 1519 1520 1525 1534 1534 1538 1539 1544 1555 1557 1561 1565 1570 1581 1581 1609 1621 1622 1625 1628 1631 1632 1633 1633 1636 1637 1643 1652 1652 1654 1658 1670 1678 1684 1694 1696 1708 1724 1730 1735 1741 1745 1746 1749 1751 1756 1765 1767 1767 1772 1780 1781 1788 1795 1807 1814 1822 1831 1838 1843 1843 1844 1855 1864 1864 1878 1882 1885 1891 1893 1893 1894 1895 1901 1902 1910 1927 1928 1935 1942 1953 1956 1957 1977 1981 1981 1984 1990 1992 1999 2000 2005 2006 2008 2014 2020 2022 2024 2029 2041 2052 2055 2064 2071 2100 2102 2104 2105 2105 2105 2111 2112 2113 2113 2128 2133 2134 2135 2135 2139 2142 2142 2167 2169 2199 2202 2210 2213 2215 2218 2219 2226 2226 2231 2238 2238 2240 2241 2247 2257 2260 2261 2264 2282 2286 2309 2312 2315 2316 2323 2325 2328 2328 2328 2343 2344 2344 2353 2367 2373 2373 2374 2375 2375 2385 2399 2399 2399 2411 2429 2438 2438 2456 2456 2457 2470 2471 2472 2472 2472 2472 2482 2486 2488 2488 2498 2514 2514 2514 2585"
    spectrum_example = list(map(int, spectrum_str.split()))
    n_example = 160
    
    best_peptide = leaderboard_cyclopeptide_sequencing(spectrum_example, n_example)
    # 输出格式：质量序列用"-"连接
    print("-".join(str(mass) for mass in best_peptide))

186-87-99-113-147-115-97-113-156-114-97-113-128-113-129-103-71-147-129-71-186-71


In [8]:
def spectral_convolution(spectrum):

    convolution = []

    n = len(spectrum)

   

    # Loop over each pair (i, j) where i > j to compute the differences

    for i in range(1, n):

        for j in range(i):

            diff = spectrum[i] - spectrum[j]

            if diff > 0:  # Only include positive differences (no zeros)

                convolution.append(diff)

   

    # Return the convolution as a list of differences with multiplicities

    return convolution

 

# Sample Input

spectrum_str = "0 137 186 323"
spectrum = list(map(int, spectrum_str.split()))

 

# Compute the convolution

convolution_result = spectral_convolution(spectrum)

 

# Output the result

print(" ".join(map(str, convolution_result)))

137 186 49 323 186 137


In [9]:
from collections import Counter

from itertools import combinations

 

def spectral_convolution(spectrum):

    convolution = []

    for i in range(len(spectrum)):

        for j in range(i):

            diff = spectrum[i] - spectrum[j]

            if 57 <= diff <= 200:

                convolution.append(diff)

    return convolution

 

def linear_spectrum(peptide):

    prefix_mass = [0]

    for mass in peptide:

        prefix_mass.append(prefix_mass[-1] + mass)

    linear_spec = [0]

    for i in range(len(prefix_mass)):

        for j in range(i + 1, len(prefix_mass)):

            linear_spec.append(prefix_mass[j] - prefix_mass[i])

    return sorted(linear_spec)

 

def cyclic_spectrum(peptide):

    prefix_mass = [0]

    for mass in peptide:

        prefix_mass.append(prefix_mass[-1] + mass)

    peptide_mass = prefix_mass[-1]

    cyclic_spec = [0]

    for i in range(len(peptide)):

        for j in range(i + 1, len(peptide) + 1):

            cyclic_spec.append(prefix_mass[j] - prefix_mass[i])

            if i > 0 and j < len(peptide):

                cyclic_spec.append(peptide_mass - (prefix_mass[j] - prefix_mass[i]))

    return sorted(cyclic_spec)

 

def score(peptide, spectrum, cyclic=True):

    if cyclic:

        pep_spec = cyclic_spectrum(peptide)

    else:

        pep_spec = linear_spectrum(peptide)

    spectrum_count = Counter(spectrum)

    pep_spec_count = Counter(pep_spec)

    return sum(min(pep_spec_count[m], spectrum_count[m]) for m in pep_spec_count)

 

def trim(leaderboard, spectrum, N):

    scored = [(pep, score(pep, spectrum, cyclic=False)) for pep in leaderboard]

    scored.sort(key=lambda x: x[1], reverse=True)

    if len(scored) <= N:

        return [x[0] for x in scored]

    threshold = scored[N-1][1]

    return [pep for pep, s in scored if s >= threshold]

 

def leaderboard_cyclopeptide_sequencing(spectrum, N, amino_acid_masses):

    leaderboard = [[]]

    leader_peptide = []

    parent_mass = max(spectrum)

    while leaderboard:

        new_leaderboard = []

        for peptide in leaderboard:

            for mass in amino_acid_masses:

                new_peptide = peptide + [mass]

                if sum(new_peptide) == parent_mass:

                    if score(new_peptide, spectrum) > score(leader_peptide, spectrum):

                        leader_peptide = new_peptide

                    new_leaderboard.append(new_peptide)

                elif sum(new_peptide) < parent_mass:

                    new_leaderboard.append(new_peptide)

        leaderboard = trim(new_leaderboard, spectrum, N)

    return leader_peptide

 

def convolution_cyclopeptide_sequencing(spectrum, M, N):

    spectrum.sort()

    conv = spectral_convolution(spectrum)

    freq = Counter(conv)

    filtered = [(mass, count) for mass, count in freq.items() if 57 <= mass <= 200]

    sorted_filtered = sorted(filtered, key=lambda x: x[1], reverse=True)

   

    result_masses = []

    if sorted_filtered:

        threshold = sorted_filtered[min(M, len(sorted_filtered))-1][1]

        for mass, count in sorted_filtered:

            if count >= threshold:

                result_masses.append(mass)

 

    return leaderboard_cyclopeptide_sequencing(spectrum, N, result_masses)

 

# Sample Input

M = 20

N = 60

spectrum_str = "57 57 71 99 129 137 170 186 194 208 228 265 285 299 307 323 356 364 394 422 493"
spectrum = list(map(int, spectrum_str.split()))

result = convolution_cyclopeptide_sequencing(spectrum, M, N)

print('-'.join(map(str, result)))

99-71-137-57-57-72


In [1]:
def middle_edge(v, w, match, mismatch, indel):
    n, m = len(v), len(w)
    mid = m // 2

    def score(a, b):
        return match if a == b else mismatch

    # Forward DP: from left to mid
    prev = [i * indel for i in range(n + 1)]
    for j in range(1, mid + 1):
        curr = [j * indel]
        for i in range(1, n + 1):
            curr.append(max(
                prev[i] + indel,                 # deletion
                curr[i - 1] + indel,             # insertion
                prev[i - 1] + score(v[i - 1], w[j - 1])  # match/mismatch
            ))
        prev = curr
    forward = prev

    # Backward DP: from right to mid
    v_rev = v[::-1]
    w_rev = w[::-1]

    prev = [i * indel for i in range(n + 1)]
    for j in range(1, m - mid + 1):
        curr = [j * indel]
        for i in range(1, n + 1):
            curr.append(max(
                prev[i] + indel,
                curr[i - 1] + indel,
                prev[i - 1] + score(v_rev[i - 1], w_rev[j - 1])
            ))
        prev = curr
    backward = prev[::-1]

    # Find best row i
    best_i = max(range(n + 1), key=lambda i: forward[i] + backward[i])

    # Determine middle edge direction
    if best_i < n and mid < m and \
       forward[best_i] + backward[best_i] == \
       forward[best_i] + score(v[best_i], w[mid]) + backward[best_i + 1]:
        return (best_i, mid), (best_i + 1, mid + 1)  # diagonal
    elif best_i < n and \
         forward[best_i] + backward[best_i] == \
         forward[best_i] + indel + backward[best_i + 1]:
        return (best_i, mid), (best_i + 1, mid)      # down
    else:
        return (best_i, mid), (best_i, mid + 1)      # right


In [3]:
v = "GAT"
w = "GAGA"
match = 1
mismatch = 1
indel = 2
middle_edge(v, w, match, mismatch, indel)

((0, 2), (1, 2))